# Experimentation with model

### Imports

In [ ]:
import sys
sys.path.append("src")  
from src.llama_activate import (
    get_pipe,
    generate_new_net,
    retrieve_existing_net,
)
import src.utils.visualization as vis
import src.utils.metrics as metrics
import src.utils.reading_in as ri
import argparse

### Set variables

In [ ]:
# Variables
rounds = 5
num_agents = 30
net = "sda"
seed = 53
alpha = 0.5
m = 2
degree = 3
p = 0.3
depressed = False
enforce_ngrams = False
use_existing_net = False
states = ["depressed", "enforce_ngrams", "basis"]
k = 0
pipe = get_pipe()
dim = 2
save = False

args = argparse.Namespace(
        net=net,
        rounds=rounds,
        num_agents=num_agents,
        seed=seed,
        m=m,
        p=p,
        k=k,
        depressed=depressed,
        enforce_ngrams=enforce_ngrams,
        alpha=alpha,
        degree = degree,
        dim = dim,
        save = save,
    )

### Generate network and run simulation

In [ ]:

networks = []
running_fracs_nets = []
fracs_dist_step_nets = []

# retrieve existing networks with different states
if use_existing_net:
    for state in states:
        args.enforce_ngrams = (state == "enforce_ngrams")
        args.depressed = (state == "depressed")
        file_path = retrieve_existing_net(args)

        # reload network from saved properties
        network, running_fracs, fracs_dist_step= ri.generate_network(file_path, pipe)
        networks.append(network)
        running_fracs_nets.append(running_fracs)
        fracs_dist_step_nets.append(fracs_dist_step)
else:
    networks, running_fracs, fracs_dist_step = generate_new_net(
        args,
        pipe,
    )
    network = networks[0][0]

#### Visualize CDS

In [ ]:
metrics.get
vis.distorted_info(network.cds_info)

#### Generate PCA plots

In [ ]:
# plot TF-IDF PCA
n_grams = metrics.load_ngrams_tsv("data/distorted_language_ngrams.tsv")
global_tf_idf, _, _ = metrics.retrieve_tf_idf(networks, num_steps=100, shift=5, n_grams=n_grams)
pca_runs = metrics.reduce_dimensionality(global_tf_idf, n_components=2)
vis.plot_tf_idf_PCA(pca_runs, states, num_steps=100, shift=5)

#### Generate plots concerning distorted info

In [ ]:
for i, network in enumerate(networks):

    print(f"Plotting results for network with state: {states[i]}")
    vis.plot_running_fracs(
        running_fracs_nets[i],
        network.m if hasattr(network, "m") else 0,
        network.p if hasattr(network, "p") else 0.0,
        enforced_ngrams=(states[i] == "enforce_ngrams"),
        depressed=(states[i] == "depressed"),
        type_nn=net,
    )
    vis.plot_distorted_fracs(
        fracs_dist_step_nets[i],
        network.m if hasattr(network, "m") else 0,
        network.p if hasattr(network, "p") else 0.0,
        enforced_ngrams=(states[i] == "enforce_ngrams"),
        depressed=(states[i] == "depressed"),
        type_nn=net,
    )


### activation plot

In [ ]:
vis.distorted_info(network.cds_info)
vis.print_network(network)
